In [2]:
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

In [3]:
DATA_PATH = Path("../data/raw")

class_names = [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
]

print("Dataset exists:", DATA_PATH.exists())
print("Training exists:", (DATA_PATH / "Train").exists())
print("Validation exists:", (DATA_PATH / "Val").exists())

Dataset exists: True
Training exists: True
Validation exists: True


In [4]:
for split in ["Train", "Val"]:
    print("\n", split)

    for name in class_names:
        folder = DATA_PATH / split / name / "images"

        images = list(folder.glob("*.jpg"))

        print(name, ":", len(images))


 Train
Glioma : 1153
Meningioma : 1449
No Tumor : 711
Pituitary : 1424

 Val
Glioma : 136
Meningioma : 140
No Tumor : 100
Pituitary : 136


In [5]:
for name in class_names:
    labels_folder = DATA_PATH / "Train" / name / "labels"

    ids = []

    for file in labels_folder.glob("*.txt"):
        for line in file.read_text().splitlines():
            parts = line.split()

            if len(parts) == 5:
                ids.append(parts[0])

    print(name, ":", dict(Counter(ids)))

Glioma : {'0': 1347, '3': 152, '1': 3}
Meningioma : {'1': 1478, '3': 103, '0': 2}
No Tumor : {'2': 711}
Pituitary : {'3': 1520, '1': 1}


In [7]:
def show_boxes(image_path):
    image_path = Path(image_path)

    label_path = (
        image_path.parent.parent
        / "labels"
        / (image_path.stem + ".txt")
    )

    image = Image.open(image_path).convert("RGB")

    draw = ImageDraw.Draw(image)

    w, h = image.size

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()

            if len(parts) != 5:
                continue

            cls, x, y, bw, bh = map(float, parts)

            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            x2 = (x + bw / 2) * w
            y2 = (y + bh / 2) * h

            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=3
            )

            draw.text(
                (x1, y1),
                str(int(cls)),
                fill="yellow"
            )

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

In [ ]:
for name in class_names:
    folder = DATA_PATH / "Train" / name / "images"

    image_path = next(folder.glob("*.jpg"))

    print(name)
    show_boxes(image_path)